# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from joblib import dump

warnings.filterwarnings('ignore')
pd.options.display.max_rows = 10

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/dayofweek.csv')
X = df.drop('dayofweek', axis=1)
y = df['dayofweek'].astype('float')
df

,user_0,user_1,user_10,user_11,user_12,user_13,user_14,user_15,user_16,user_17,...,lab05s,laba04,laba04s,laba05,laba06,laba06s,project1,numTrials,hour,dayofweek
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.788667,-2.562352,4
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.756764,-2.562352,4
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.724861,-2.562352,4
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.692958,-2.562352,4
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.661055,-2.562352,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.533442,0.945382,3
1682,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.629151,0.945382,3
1683,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.597248,0.945382,3
1684,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.565345,0.945382,3


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [4]:
def crossval(n_splits: int, X: pd.DataFrame, y: pd.DataFrame, model) -> str:
    skf = StratifiedKFold(n_splits=n_splits, random_state=21, shuffle=True)
    out = ""; acc_list: list = []
    # get split indexes 
    for train_ix, test_ix in skf.split(X, y):
        y_model = y.iloc[train_ix]; y_test = y.iloc[test_ix]
        X_model = X.iloc[train_ix]; X_test = X.iloc[test_ix]
        X_train, X_valid, y_train, y_valid = train_test_split(X_model, y_model, test_size=0.25, random_state=21)

        model.fit(X_train, y_train)

        for x_split, y_split, text in ((X_train, y_train, 'train - {:.5f} | '), (X_valid, y_valid, 'valid - {:.5f}\n')):
            predict_col = model.predict(x_split)
            acc = accuracy_score(predict_col, y_split)
            if text == 'valid - {:.5f}\n':
                acc_list.append(acc)
            out += text.format(acc)

    out += f'Average accuracy on crossval is {np.mean(acc_list):.5f}\n'
    out += f'Std is {np.std(acc_list):.5f}'
    return out

In [5]:
%%time

log_reg = LogisticRegression(random_state=21, fit_intercept=False)
log_reg_crossval = crossval(10, X, y, log_reg)
print(log_reg_crossval)

train - 0.64116 | valid - 0.60000
train - 0.65435 | valid - 0.65789
train - 0.62973 | valid - 0.58158
train - 0.63149 | valid - 0.63421
train - 0.65611 | valid - 0.63158
train - 0.63764 | valid - 0.58947
train - 0.63972 | valid - 0.59211
train - 0.63533 | valid - 0.55789
train - 0.63093 | valid - 0.59737
train - 0.65554 | valid - 0.63158
Average accuracy on crossval is 0.60737
Std is 0.02875
CPU times: user 3.29 s, sys: 22.4 ms, total: 3.31 s
Wall time: 505 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [6]:
for solver in ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']:
    log_reg = LogisticRegression(random_state=21, fit_intercept=False, solver=solver)
    log_reg_crossval = crossval(10, X, y, log_reg)
    print(f'{solver=}', log_reg_crossval[-55:])

    for penalty in [None, 'l1', 'l2']:
        try:
            log_reg = LogisticRegression(random_state=21, fit_intercept=False, solver=solver, penalty=penalty)
            log_reg_crossval = crossval(10, X, y, log_reg)
            print(f'{solver=} {penalty=}', log_reg_crossval[-55:])
        except ValueError:
            pass

solver='lbfgs' 
Average accuracy on crossval is 0.60737
Std is 0.02875
solver='lbfgs' penalty=None 
Average accuracy on crossval is 0.63421
Std is 0.02516
solver='lbfgs' penalty='l2' 
Average accuracy on crossval is 0.60737
Std is 0.02875
solver='liblinear' 
Average accuracy on crossval is 0.58500
Std is 0.02199
solver='liblinear' penalty='l1' 
Average accuracy on crossval is 0.59132
Std is 0.02239
solver='liblinear' penalty='l2' 
Average accuracy on crossval is 0.58500
Std is 0.02199
solver='newton-cg' 
Average accuracy on crossval is 0.60763
Std is 0.02862
solver='newton-cg' penalty=None 
Average accuracy on crossval is 0.63553
Std is 0.02369
solver='newton-cg' penalty='l2' 
Average accuracy on crossval is 0.60763
Std is 0.02862
solver='newton-cholesky' 
Average accuracy on crossval is 0.60737
Std is 0.02870
solver='newton-cholesky' penalty=None 
Average accuracy on crossval is 0.63474
Std is 0.02537
solver='newton-cholesky' penalty='l2' 
Average accuracy on crossval is 0.60737
Std i

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [7]:
%%time

svc = SVC(random_state=21, probability=True, kernel='linear')
svc_crossval = crossval(10, X, y, svc)
print(svc_crossval)

train - 0.70009 | valid - 0.65526
train - 0.68953 | valid - 0.68684
train - 0.68865 | valid - 0.62368
train - 0.69833 | valid - 0.63684
train - 0.71152 | valid - 0.64737
train - 0.70976 | valid - 0.65789
train - 0.72671 | valid - 0.65000
train - 0.70123 | valid - 0.62632
train - 0.70387 | valid - 0.67105
train - 0.71265 | valid - 0.66842
Average accuracy on crossval is 0.65237
Std is 0.01899
CPU times: user 2.53 s, sys: 3.24 ms, total: 2.54 s
Wall time: 2.54 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [8]:
for c in range(1, 7, 1):
    svc = SVC(random_state=21, probability=True, kernel='linear', C=c)
    svc_crossval = crossval(10, X, y, svc)
    print(f'{c=}', svc_crossval[-55:])

c=1 
Average accuracy on crossval is 0.65237
Std is 0.01899
c=2 
Average accuracy on crossval is 0.66500
Std is 0.01393
c=3 
Average accuracy on crossval is 0.68342
Std is 0.01905
c=4 
Average accuracy on crossval is 0.68921
Std is 0.02100
c=5 
Average accuracy on crossval is 0.68921
Std is 0.02019
c=6 
Average accuracy on crossval is 0.69395
Std is 0.01966


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [9]:
%%time

dec_tree_class = DecisionTreeClassifier(random_state=21, max_depth=10)
dec_tree_class_crossval = crossval(10, X, y, dec_tree_class)
print(dec_tree_class_crossval)

train - 0.84697 | valid - 0.76579
train - 0.84257 | valid - 0.76842
train - 0.83641 | valid - 0.77105
train - 0.81354 | valid - 0.74737
train - 0.85048 | valid - 0.73947
train - 0.82146 | valid - 0.73158
train - 0.85062 | valid - 0.75000
train - 0.85501 | valid - 0.71053
train - 0.84359 | valid - 0.78947
train - 0.81898 | valid - 0.73684
Average accuracy on crossval is 0.75105
Std is 0.02183
CPU times: user 129 ms, sys: 4.95 ms, total: 134 ms
Wall time: 133 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [10]:
for depth in range(5, 13, 1):
    dec_tree_class = DecisionTreeClassifier(random_state=21, max_depth=depth)
    dec_tree_class_crossval = crossval(10, X, y, dec_tree_class)
    print(f'{depth=}', dec_tree_class_crossval[-55:])

depth=5 
Average accuracy on crossval is 0.58289
Std is 0.02168
depth=6 
Average accuracy on crossval is 0.62947
Std is 0.02166
depth=7 
Average accuracy on crossval is 0.65184
Std is 0.02676
depth=8 
Average accuracy on crossval is 0.69158
Std is 0.02034
depth=9 
Average accuracy on crossval is 0.72316
Std is 0.01709
depth=10 
Average accuracy on crossval is 0.75105
Std is 0.02183
depth=11 
Average accuracy on crossval is 0.77447
Std is 0.01800
depth=12 
Average accuracy on crossval is 0.79789
Std is 0.02000


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [11]:
%%time

random_forest = RandomForestClassifier(n_estimators=50, random_state=21, max_depth=14)
random_forest_crossval = crossval(10, X, y, random_forest)
print(random_forest_crossval)

train - 0.97098 | valid - 0.86053
train - 0.97010 | valid - 0.88947
train - 0.95163 | valid - 0.86579
train - 0.97010 | valid - 0.89211
train - 0.97713 | valid - 0.86053
train - 0.97801 | valid - 0.89474
train - 0.96134 | valid - 0.85526
train - 0.97452 | valid - 0.83947
train - 0.95606 | valid - 0.88421
train - 0.96485 | valid - 0.87632
Average accuracy on crossval is 0.87184
Std is 0.01742
CPU times: user 1.03 s, sys: 4.03 ms, total: 1.03 s
Wall time: 1.03 s


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [12]:
for depth in range(1, 21, 4):
    for n_estimators in range(1, 76, 24):
        random_forest = RandomForestClassifier(n_estimators=n_estimators, random_state=21, max_depth=depth)
        random_forest_crossval = crossval(10, X, y, random_forest)
        print(f'{depth=} {n_estimators=}', random_forest_crossval[-55:])

depth=1 n_estimators=1 
Average accuracy on crossval is 0.27184
Std is 0.00833
depth=1 n_estimators=25 
Average accuracy on crossval is 0.38684
Std is 0.01470
depth=1 n_estimators=49 
Average accuracy on crossval is 0.39447
Std is 0.01082
depth=1 n_estimators=73 
Average accuracy on crossval is 0.40053
Std is 0.01621
depth=5 n_estimators=1 
Average accuracy on crossval is 0.44842
Std is 0.01363
depth=5 n_estimators=25 
Average accuracy on crossval is 0.57053
Std is 0.03804
depth=5 n_estimators=49 
Average accuracy on crossval is 0.58237
Std is 0.02276
depth=5 n_estimators=73 
Average accuracy on crossval is 0.57947
Std is 0.02540
depth=9 n_estimators=1 
Average accuracy on crossval is 0.60237
Std is 0.02685
depth=9 n_estimators=25 
Average accuracy on crossval is 0.75711
Std is 0.02589
depth=9 n_estimators=49 
Average accuracy on crossval is 0.76605
Std is 0.02087
depth=9 n_estimators=73 
Average accuracy on crossval is 0.76474
Std is 0.01742
depth=13 n_estimators=1 
Average accuracy o

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [13]:
model = RandomForestClassifier(n_estimators=73, random_state=21, max_depth=17)
model.fit(X_train, y_train)
predict_col = model.predict(X_test)
acc = accuracy_score(predict_col, y_test)
acc

0.908284023668639

In [14]:
y_test_compare = y_test.reset_index(drop=True).value_counts().sort_values()
predict_compare = pd.Series(predict_col).value_counts().sort_values()
compare = pd.concat([predict_compare, y_test_compare], keys=['predict', 'as_is'], axis=1)
compare['%_error'] = abs(compare['predict'].values / compare['as_is'].values - 1 )
compare.sort_values(by='%_error', ascending=False)

,predict,as_is,%_error
0.0,22,27,0.185185
1.0,48,55,0.127273
3.0,90,80,0.125000
4.0,19,21,0.095238
5.0,59,54,0.092593
2.0,28,30,0.066667
6.0,72,71,0.014085


In [15]:
with open("00_regularization.joblib", "wb") as f:
    dump(model, f, protocol=-1)
    # The optional protocol argument, an integer, tells the pickler to use the given protocol; 
    # supported protocols are 0 to HIGHEST_PROTOCOL. If not specified, the default is DEFAULT_PROTOCOL. 
    # If a negative number is specified, HIGHEST_PROTOCOL is selected.